# Phase 5 · Legal statute identification · new GQA causal decoder (4B tokens), full fine-tuning
Identical recipe for the three new pretrained models (seed 17, same chunks, same schedule, 4 x 12,000 documents). Only the checkpoint differs, so the dev scores compare the models. Needs `hybrid.py` and `encoder.py` next to `rr_lib.py` in `MyDrive/legal-llm-course/` and the finished `model_final.pt` of the matching pretraining run.

In [ ]:
# 1 · Settings
from pathlib import Path
ACTION   = 'Train'          # Inspect | Train | Predict   (Predict = reload best.pt and rewrite the submissions)
PRESET   = 'NEW-GQA'    # GQA-prenorm | MHA-prenorm | MHA-postnorm | MLA-prenorm
METHOD   = 'full'           # full | partial | lora | rslora | lorma | ia3 | kd
RUN_ID   = 'new-gqa-full-001'
SEED     = 17

# long-document representation
CHUNK_LEN, STRIDE = 512, 512        # tokens per chunk
MAX_CHUNKS = 8                      # chunks per document used in training/inference (8 chunks cover about 90% of documents fully)
SELECT   = 'uniform'      # first | last | uniform | head_tail | random
AGG      = 'attn'            # mean | attn | bigru      (how chunk vectors are pooled into a document vector)
POOL     = 'meanlast'    # meanlast | meanmaxlast     (meanmaxlast adds max-pooling over each chunk's tokens: keeps sparse cues)
MAX_TOKENS = 40000

# multi-label loss
CLASS_WEIGHT = 'sqrt'               # none | sqrt | inverse   (positive-class weights in the BCE loss, per statute)
FOCAL_GAMMA  = 0.0                  # > 0 adds a focal term to the BCE

# budget: each epoch trains on DOCS_PER_EPOCH random documents (of ~42,000); per-epoch evaluation uses DEV_SUBSET dev documents
EPOCHS, DOCS_PER_EPOCH, MAX_MINUTES = 4, 12000, 120   # MAX_MINUTES stops mid-epoch when crossed (best.pt is kept)
BATCH_DOCS, ACCUM, EVAL_DOCS, DEV_SUBSET, LOG_EVERY = 4, 4, 16, 3000, 100

COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_5_LSI'
RUNS_ROOT   = COURSE_ROOT/'runs/phase5_lsi'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'   # legal_scaffold, training_methods, tokenizer
RR_LIB, CJPE_LIB, LSI_LIB = COURSE_ROOT/'rr_lib.py', COURSE_ROOT/'cjpe_lib.py', COURSE_ROOT/'lsi_lib.py'

# adapter / method knobs
RANK, ALPHA = 8, 16
LORA_TARGETS = ('query', 'value')
PARTIAL_LAYERS = (10, 11)
# knowledge distillation (METHOD='kd'): frozen teacher = a finished `full` run on the same chunks
KD_TEACHER_RUN = RUNS_ROOT/'MHA-prenorm/full/mha-full-001'
KD_STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.3

DEFAULT_LR = {  # backbone / adapter / head
 'full': (5e-5, 0, 1e-3), 'kd': (5e-5, 0, 1e-3), 'partial': (1e-4, 0, 1e-3),
 'lora': (0, 1e-3, 1e-3), 'rslora': (0, 1e-3, 1e-3), 'lorma': (0, 1e-3, 1e-3), 'ia3': (0, 3e-3, 1e-3)}
assert ACTION in ('Inspect','Train','Predict') and METHOD in DEFAULT_LR and SELECT in ('first','last','uniform','head_tail','random') and POOL in ('meanlast','meanmaxlast')
PRESETS = {
 'GQA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='gqa',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-postnorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='post',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MLA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='mla',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02,mla_latent=256,mla_query_rank=256,mla_rope_dim=32)}

# the new checkpoints carry their own config and objective (rr_lib.load_backbone), so these entries only name the preset
for _k in ('NEW-GQA', 'NEW-ENC', 'NEW-HYB'): PRESETS[_k] = PRESETS['GQA-prenorm']

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py'); shutil.copy(LSI_LIB, LOCAL/'lsi_lib.py')
for _f in ('hybrid.py', 'encoder.py'): shutil.copy(COURSE_ROOT/_f, LOCAL/_f)   # rr_lib.load_backbone imports these for the hybrid / encoder checkpoints
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J, lsi_lib as LSI
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data: tokenised documents come from a cache on Drive (build once with tools/build_token_cache.py, or the first run builds it)
CACHE = PHASE_DIR/f'tokens_lsi_v1_max{MAX_TOKENS}.npz'
DATA = LSI.load_cache(CACHE, TOKENIZER_SHA256, MAX_TOKENS)
if DATA is None:
    from google.colab import userdata
    try: token = userdata.get('HF_TOKEN')
    except Exception: token = None
    LSI.ensure_files(COURSE_ROOT, token)
    DATA = LSI.load_all(PHASE_DIR)
    for s in DATA.values(): J.tokenize_docs(s, TOK, MAX_TOKENS)
    print('saved cache:', LSI.save_cache(DATA, CACHE, TOKENIZER_SHA256, MAX_TOKENS))
else:
    print('loaded tokenised documents from', CACHE)
for s, docs in DATA.items():
    n = np.array([len(d['tok']) for d in docs]); k = np.array([len(d['labels']) for d in docs]) if docs[0]['labels'] is not None else None
    print(f"{s:5s}: {len(docs):6d} documents | tokens mean/median/p90 {int(n.mean())}/{int(np.median(n))}/{int(np.percentile(n,90))}" + (f" | statutes per document mean {k.mean():.2f} max {k.max()}" if k is not None else ' | no labels'))
ids = [d['id'] for d in DATA['test']]
assert len(ids) == len(set(ids)), 'duplicate test ids'
assert all(d['labels'] for d in DATA['train'] + DATA['dev']), 'every train/dev document should have at least one statute'
assert all(d['labels'] is None for d in DATA['test']), 'test labels must not be loaded'
print('chunks per train document (mean):', round(np.mean([len(J.chunk_spans(len(d['tok']), CHUNK_LEN, STRIDE)) for d in DATA['train'][:5000]]), 1), '| MAX_CHUNKS =', MAX_CHUNKS)

In [ ]:
# 4 · Build run config + model from the Phase 1 snapshot
lb, la, lh = DEFAULT_LR[METHOD]
C = dict(method=METHOD, preset=PRESET, run_id=RUN_ID, class_weight=CLASS_WEIGHT, focal_gamma=FOCAL_GAMMA, epochs=EPOCHS, seed=SEED,
         lr=dict(backbone=lb, adapter=la, head=lh), partial_layers=PARTIAL_LAYERS,
         chunk_len=CHUNK_LEN, stride=STRIDE, max_chunks=MAX_CHUNKS, select=SELECT, agg=AGG, pool=POOL,
         docs_per_epoch=DOCS_PER_EPOCH, max_minutes=MAX_MINUTES, batch_docs=BATCH_DOCS, accum=ACCUM, eval_docs=EVAL_DOCS, dev_subset=DEV_SUBSET, log_every=LOG_EVERY,
         kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
         head_kw=dict(n_classes=LSI.N_LABELS, hidden=256, agg=AGG, dropout=0.1, pool=POOL),
         adapter_kw=dict(rank=RANK, alpha=ALPHA, targets=LORA_TARGETS),
         layer_keep=KD_STUDENT_LAYERS if METHOD=='kd' else None)
RUN_ROOT = RUNS_ROOT/PRESET/METHOD/RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

PHASE1_OVERRIDE = {'MHA-prenorm': str(COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt'), 'MHA-postnorm': str(COURSE_ROOT/'runs/mha_post/run-001/legal-token-stages-v1/run-001-milestone-60762-53fcafd4/state.pt')}

def _scan_snapshots(preset):
    """Diagnostic: list every snapshot.json under runs/ (follows Drive shortcuts) vs the wanted config."""
    import os
    from legal_scaffold.models import ModelConfig
    want = ModelConfig(**PRESETS[preset]).validate().__dict__
    rows = []
    for dp, dn, fn in os.walk(COURSE_ROOT/'runs', followlinks=True):
        if 'snapshot.json' in fn:
            try:
                m = json.loads((Path(dp)/'snapshot.json').read_text())
                ident = m.get('identity', {}); cfg = ident.get('model', ident.get('config')) or {}
                got = ModelConfig(**cfg).validate().__dict__ if cfg else {}
                st = m.get('state_inventory', {}).get('model')
                rows.append((got == want, m.get('stage'), m.get('progress', {}).get('step'), bool(st and (Path(dp)/st).is_file()), dp))
            except Exception as e:
                rows.append((False, 'ERR '+str(e)[:40], None, False, dp))
    return rows

def phase1_checkpoint(preset):
    if preset in PHASE1_OVERRIDE:
        return Path(PHASE1_OVERRIDE[preset]), {'stage': 'override'}
    for kw in (dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=TOKENIZER_SHA256),
               dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=None),
               dict(stages=('legal_continued','legal_only','wikitext_only','legal','pretrain'), tokenizer_sha256=None)):
        try:
            return resolve_snapshot_checkpoint(COURSE_ROOT/'runs', PRESETS[preset], **kw)
        except FileNotFoundError:
            pass
    rows = _scan_snapshots(preset)
    print(f'--- No usable Phase 1 snapshot for {preset}. {len(rows)} snapshot.json files found under runs/ ---')
    for r in sorted(rows, key=lambda r: (not r[0], str(r[1])))[:25]:
        print('config_match=%s stage=%s step=%s payload_present=%s  %s' % r)
    raise FileNotFoundError(f'{preset}: no snapshot with matching config + payload; set PHASE1_OVERRIDE[{preset!r}] to its state.pt')



PHASE1_OVERRIDE.update({'NEW-GQA': str(COURSE_ROOT/'pretrain/pretrain_dec_gqa/model_final.pt'), 'NEW-ENC': str(COURSE_ROOT/'pretrain/pretrain_enc_mlm/model_final.pt'), 'NEW-HYB': str(COURSE_ROOT/'pretrain/pretrain_hybrid/model_final.pt')})

def build(C):
    ck, man = phase1_checkpoint(C['preset'])
    m = J.build_model(PRESETS[C['preset']], ck, 'full' if C['method']=='kd' else C['method'], head_kw=C['head_kw'],
                      adapter_kw=C['adapter_kw'] if C['method'] not in ('full','kd','partial') else {},
                      layer_keep=C['layer_keep'])
    return m, ck

if ACTION == 'Inspect':
    for p in PRESETS:
        try: print(p, '->', phase1_checkpoint(p)[0])
        except FileNotFoundError as e: print(p, '-> NOT FOUND')
    raise SystemExit('Inspect only.')
model, ckpt = build(C)
print('Phase 1 snapshot:', ckpt)
tr, tot = R.set_trainable(model, METHOD, C['partial_layers'])
print(f'trainable {tr:,} / {tot:,} ({100*tr/tot:.2f}%)   run folder: {RUN_ROOT}')

teacher = None
if METHOD == 'kd':
    tsaved = torch.load(KD_TEACHER_RUN/'best.pt', map_location='cpu', weights_only=False)
    tC = tsaved['C']
    assert tC['method'] == 'full', 'teacher should be a finished full fine-tune'
    for k in ('chunk_len','stride','max_chunks'): assert tC[k] == C[k], f'teacher/student must share {k}'
    teacher, _ = build(tC)
    teacher.load_state_dict(tsaved['trainable'], strict=False)
    teacher.to(device).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    n_t = sum(p.numel() for p in teacher.parameters()); n_s = sum(p.numel() for p in model.parameters())
    assert n_t > n_s, 'teacher must be larger than student'
    print(f'teacher {tC["preset"]} {n_t:,} params -> student {PRESET}/{len(KD_STUDENT_LAYERS)} layers {n_s:,} params')

In [ ]:
# 5 · Train (resumes automatically from last.pt after a Colab disconnect)
if ACTION == 'Train':
    RESULTS, AMP, P_DEV = LSI.train_run(model, DATA, RUN_ROOT, C, device, teacher=teacher)
else:
    saved = torch.load(RUN_ROOT/'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(saved['trainable'], strict=False); model.to(device)
    AMP = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16
    RESULTS = json.loads((RUN_ROOT/'results.json').read_text()); P_DEV = None
print('best epoch', RESULTS['best_epoch'], '| trainable', f"{RESULTS['trainable_params']:,}", '| train minutes', round(RESULTS['train_minutes'] or 0, 1))

In [ ]:
# 6 · Score dev + test; write both submissions and the probability files (for ensembling)
import pandas as pd
EV, P_DEV, P_TEST, OUT = LSI.export_run(model, DATA, C, device, AMP, RUN_ROOT, P_dev=P_DEV)
rows = [{'threshold strategy': k, **{m: round(v[m], 4) for m in ('macro_f1', 'micro_f1', 'subset_acc', 'macro_precision', 'macro_recall', 'labels_per_doc_pred')}} for k, v in EV.items() if isinstance(v, dict)]
display(pd.DataFrame(rows)); print('global threshold:', EV['global_threshold'])
print('upload one file, as-is:', {k: v['zip'] for k, v in OUT.items()})

In [ ]:
# 7 · Analysis for the report: rare statutes, thresholds, confusions, label relations, sections, cost
Y = LSI.multi_hot(DATA['dev']); g = EV['global_threshold']
tab = pd.DataFrame(LSI.per_label_table(Y, P_DEV, g, DATA['train']))
print('five hardest / five easiest statutes (global threshold, dev):'); display(pd.concat([tab.sort_values('F1').head(5), tab.sort_values('F1').tail(5)]))
tab['train-frequency quartile'] = pd.qcut(tab.train_docs.rank(method='first'), 4, labels=['Q1 rarest', 'Q2', 'Q3', 'Q4 most common'])
print('mean F1 by how common the statute is in training:'); print(tab.groupby('train-frequency quartile', observed=True).F1.mean().round(4))
print('rank correlation of per-statute F1 with training frequency:', round(tab.F1.corr(tab.train_docs, method='spearman'), 3))
pred = LSI.decide(P_DEV, g, True)
print('\nstatute pairs most often confused (missed one, predicted another, same document):'); display(pd.DataFrame(LSI.confusion_pairs(Y, pred, 12)))
card = Y.sum(1); bucket = pd.cut(card, [0, 1, 2, 4, 8, 100], labels=['1', '2', '3-4', '5-8', '9+'])
print('exact-match accuracy and micro-F1 by number of true statutes in the document:')
display(pd.DataFrame([{'true statutes': b, 'documents': int((bucket == b).sum()), 'exact match': round(float((pred[bucket == b] == Y[bucket == b].astype(bool)).all(1).mean()), 3)} for b in bucket.categories]))
S2 = LSI.stack_cooccurrence_cv(Y, P_DEV)
print('\nDo label relationships help? second-stage model on the 100 probabilities (cross-validated on dev):')
print('  independent predictions, global threshold:', round(LSI.scores(Y, LSI.decide(P_DEV, g, True))['macro_f1'], 4), '| stacked on label probabilities:', round(LSI.scores(Y, LSI.decide(S2, LSI.tune_global(Y, S2)[0], True))['macro_f1'], 4))
print('\nwhich part of a document matters? (this model, other chunk selections, 2,000 dev documents, no retraining)')
abl = pd.DataFrame(LSI.selection_ablation(model, DATA['dev'][:2000], C, device, AMP)).pivot(index='select', columns='chunks', values='macro_f1_global_tuned'); display(abl)
print(f"\ncost: {RESULTS['total_params']/1e6:.1f}M params, {RESULTS['trainable_params']/1e6:.2f}M trainable, peak GPU {RESULTS['peak_gpu_gb']} GB, "
      f"train {RESULTS['train_minutes']:.1f} min, inference {RESULTS['infer_sec_per_doc']*1000:.0f} ms/document")
print(pd.DataFrame(RESULTS['history'])[['epoch', 'train_loss', 'dev_macro_f1_tuned', 'dev_micro_f1_tuned', 'minutes']].round(4))

In [ ]:
# 8 · Compare every finished run (for the report's tables)
rows = []
for f in sorted(RUNS_ROOT.rglob('results.json')):
    r = json.loads(f.read_text()); d = r['dev']
    rows.append({'run': f.parent.name, 'preset': r['preset'], 'method': r['method'], 'select': r['setup']['select'], 'agg': r['setup']['agg'], 'weights': r['setup']['class_weight'],
                 'trainable_M': round(r['trainable_params']/1e6, 3), 'total_M': round(r['total_params']/1e6, 1), 'best_ep': r['best_epoch'],
                 'macro_F1 (global)': round(d['global_tuned+min_one']['macro_f1'], 4), 'macro_F1 (per-label, cross-fit)': round(d['per_label_cross_fit+min_one']['macro_f1'], 4),
                 'micro_F1': round(d['global_tuned+min_one']['micro_f1'], 4), 'subset_acc': round(d['global_tuned+min_one']['subset_acc'], 4),
                 'min': round(r['train_minutes'] or 0, 1), 'peak_GB': r['peak_gpu_gb'] and round(r['peak_gpu_gb'], 1), 'ms/doc': round(r['infer_sec_per_doc']*1000)})
if rows: display(pd.DataFrame(rows).sort_values('macro_F1 (global)', ascending=False))